# 🎙️ Voice Clone Studio — run on Google Colab (free GPU)

Runs the Flask + Chatterbox TTS app on Colab's free GPU and gives you a public URL.

**Before you start:** set the runtime to GPU →
`Runtime` ▸ `Change runtime type` ▸ Hardware accelerator: **T4 GPU** ▸ Save.

Then edit `REPO_URL` in the next cell and run all cells top to bottom.
The last cell prints a `https://....trycloudflare.com` link — that's your app.
Keep this tab open; the link dies when the Colab session ends.

In [ ]:
# 1) Clone your repo
REPO_URL = "https://github.com/Lintagni/voice-clone.git"

import os
%cd /content
!rm -rf voice-clone
!git clone $REPO_URL voice-clone
%cd /content/voice-clone
print(os.listdir('.'))

In [ ]:
# 2) Install dependencies
!pip install -q -r requirements.txt
# Ensure a CUDA build of torch so generation uses the GPU (fast).
!pip install -q torch==2.6.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 3) Get cloudflared (free, no-signup tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print('cloudflared ready')

In [ ]:
# 4) Launch the app + open the public tunnel
import os, subprocess, socket, time, re

os.environ['PORT'] = '5000'
flask_proc = subprocess.Popen(['python', 'app.py'], cwd='/content/voice-clone')

def wait_port(port, timeout=180):
    t = time.time()
    while time.time() - t < timeout:
        try:
            with socket.create_connection(('127.0.0.1', port), 2):
                return True
        except OSError:
            time.sleep(1)
    return False

print('Starting Flask server...')
assert wait_port(5000), 'Flask did not start — check the cell-2 install output.'
print('Server up. Opening tunnel...\n')

tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in tunnel.stdout:
    m = re.search(r'https://[-\w.]+trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print('\n' + '=' * 60)
print('✅ OPEN YOUR APP HERE:', public_url)
print('=' * 60)
print('(First generation downloads the model — give it a minute.)')